# 01 · Read temperature and status

Read temperature, setpoint and diagnostics. Readings are sequential, not simultaneous; compare with the front panel and an independent reference.

Run top to bottom in a **Python 3.12+** kernel. This notebook includes all configuration and setup; no sibling example files are imported. Run only one hardware notebook/process at a time.

**Prepare the filled cooling loop before connecting.** T255P connection sends U, takes serial control and can change AUTO START to RUN, even with target writes disabled. Software disconnect does not stop the pump/chiller or restore its target. Physical hardware has not been validated with this package.

## 1. Environment and imports

From the repository root, run `uv sync --locked --group notebooks`, then `uv run --locked --group notebooks jupyter lab`. Select the environment kernel described in examples/README.md. The script-only dashboard (06_dashboard.py) needs `--extra gui`. Restart the kernel after installation or driver source changes. Imports open no serial ports.

In [1]:
import sys
from pathlib import Path

# Prefer checkout sources when launched from the repository or examples folder.
for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / "src/tark_chiller/controller.py").is_file():
        sys.path.insert(0, str(candidate / "src"))
        break

In [2]:
from tark_chiller.controller import T255P, TarkChiller

## 2. Choose the port

This example connects to a **ThermoTek T255P**. Set `PORT` to its serial port (for example `"COM9"`). Adapter metadata identifies the USB adapter, not the attached chiller.

The connection cell below spells out the T255P profile: 9600 baud, 8N1, XON/XOFF, no hardware flow control, and RTS/DTR deasserted. Check that your adapter/cable supports these line levels. Distilled-water target bounds are 5–45 °C, in 0.1 °C steps. See the [hardware guide](../docs/hardware.md) for wiring and the separate CAL protocol.

In [3]:
PORT = "COM9"  # Set the identified serial port, e.g. "COM9".

## 3. Connection settings

The next operation cell constructs the driver directly with the settings shown. Construction opens nothing. Its context connects and releases the port, including on a read error or interrupt. Among the notebooks, only example 02 enables target writes; dashboard script 06 also enables explicit GUI target requests.

## 4. Read temperature and status

Read temperature, setpoint and diagnostics. Readings are sequential, not simultaneous; compare with the front panel and an independent reference.

Run the next cell as a unit: it owns connection and cleanup together. Interrupt the cell to cancel; errors remain visible.

In [4]:
if not isinstance(PORT, str) or not PORT.strip():
    raise ValueError('Hardware not configured: set PORT, e.g. "COM9". No port was opened.')

chiller = TarkChiller(
    {
        "port": PORT,
        "baudrate": 9600,
        "bytesize": 8,
        "parity": "N",
        "stopbits": 1,
        "xonxoff": True,
        "rtscts": False,
        "dsrdtr": False,
        "rts": False,
        "dtr": False,
    },
    protocol=T255P(allow_writes=False),
    setpoint_range=(5.0, 45.0),
    coolant_source="ThermoTek T255P Rev C pp9/20; verify installed distilled water",
)

# T255P connect sends U: it takes serial control and can start AUTO START.
with chiller:
    # 2. Query the measured temperature and target separately, in Celsius.
    # Compare these values with the front panel and your temperature reference.
    print(f"Temperature:       {chiller.read_temperature():.2f} C")
    print(f"Reported setpoint: {chiller.read_setpoint():.2f} C")

    # 3. Inspect mode and alarm diagnostics; these do not identify the unit.
    status = chiller.read_status()
    print(f"Connected: {status['connected']}")
    print(f"Backend:   {status['backend']}")
    print(f"Details:   {status['detail']}")

# 4. Context exit releases the port, including after a failed read. If cleanup
# also fails, the original error is preserved with a cleanup note. Retry cleanup
# as needed; closing the software connection does not stop equipment.

Temperature:       17.99 C
Reported setpoint: 18.00 C
Connected: True
Backend:   hardware
Details:   T255P (human-selected; identity not queried); mode=RUN; summary_alarm=0 chiller_on=1 dryer_on=0; reported alarms: float_switch=0 high_temp=0 low_temp=0 sensor=0 eeprom=0 watchdog=0; flow/leak unknown; last serial reply, not safety proof


## 5. Check results and finish

T255P diagnostics report mode/alarm flags; they do not identify the unit.

Context cleanup closes software resources, including after a read error or kernel interrupt. If cleanup fails, resources may remain owned: allow pending I/O to finish and retry `chiller.disconnect(timeout_s=5.0)` if `chiller` was created. Do not rerun configuration over a live connection. A forced kernel termination cannot guarantee cleanup. Follow the equipment's physical shutdown procedure separately.